# PolicyShield on Google Colab (free tier)
Policy-grounded multi-agent response to a telecom DDoS when the AI detector fails silently.
ITU AI for Good Lab Hackathon (UAE) 2026. Runs on the free Colab tier with no API key.

**Runtime:** select *Runtime > Change runtime type > T4 GPU* (CPU also works, but more slowly).

In [ ]:
!git clone https://github.com/Bereket13/telecom-ddos-policy-agents.git
%cd telecom-ddos-policy-agents
!pip -q install faiss-cpu pypdf sentence-transformers huggingface_hub
!pip -q install llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

## 1. Knowledge base
The prebuilt index is in `kb/index`. To rebuild it from the original documents, run the next cell.

In [ ]:
REBUILD = False
if REBUILD:
    !python src/download_sources.py docs
    !python src/build_kb.py --docs docs --out kb/index

## 2. Model (Qwen2.5-3B-Instruct, 4-bit GGUF, about 1.9 GB)

In [ ]:
import os
from huggingface_hub import hf_hub_download
MODEL = os.path.abspath(hf_hub_download('bartowski/Qwen2.5-3B-Instruct-GGUF', 'Qwen2.5-3B-Instruct-Q4_K_M.gguf', local_dir='models'))
MODEL

## 3. Run the chunked error scenario (4 rounds, 4 agents + coordinator)

In [ ]:
%cd src
!python run_scenario.py --kb ../kb/index --model {MODEL} --scenario ../scenarios/ddos_mobile_operator.json --out ../outputs
%cd ..

## 4. Inspect the audit trail

In [ ]:
import glob, json
run = sorted(glob.glob('outputs/run_*'))[-1]
for line in open(f'{run}/audit_log.jsonl', encoding='utf-8'):
    r = json.loads(line)
    print(r['round'], r['sim_time'], r['agent'], r['type'], [c['status'] for c in r.get('citation_check', [])])
print(open(f'{run}/transcript.md', encoding='utf-8').read()[:3000])